# 독립 검증 노트북 — "계산이 맞는가"

**목적**: 프로젝트 코드(`drone_lib.py` · `scripts/check.py` · 스킬 스크립트)를 **import하지 않고**, pandas · numpy · scikit-learn만으로
AGENTS.md 규칙을 처음부터 다시 구현해 저장된 결과 파일과 대조한다.

- 기대값은 AGENTS.md 규칙 · `artifacts/model_card.json` · `plan_meta.json` · `evaluation.json`에서 가져온다.
- 원본 `data/`와 결과 파일은 **읽기만** 한다. 결함 주입(12번)은 메모리 복사본에서만 한다.
- 이 노트북은 '계산이 맞는가'만 본다. '예측이 맞았는가'(2026-01 20대 적중)는 2026-02 기록이 있어야 알 수 있다.

| # | 점검 |
|---|---|
| 1 | 전처리 규칙 |
| 2 | 시간 피처 · 계절 |
| 3 | 누수 컬럼 제외 |
| 4 | 기간 분할 건수 |
| 5 | 검증 성능 · 월 20대 적중 |
| 6 | 임계값 선택 |
| 7 | 시험 예측(제출물 1) |
| 8 | 2025 평가(정답 재현) |
| 9 | 재점검 20대(score_2026-01) |
| 10 | 앱 데이터(Supabase) = CSV |
| 11 | 지문(SHA-256 앞 12자리) 연결 |
| 12 | 결함 주입 — 점검이 실제로 잡는가 |

In [1]:
import hashlib, json, tomllib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
DATA, ART, OUT = ROOT / "data", ROOT / "artifacts", ROOT / "outputs"

# 'N/A'(예방정비 고장 없음)를 pandas가 결측으로 바꾸지 않도록 빈칸만 결측으로 읽는다
def read(path):
    return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, na_values=[""])

card = json.loads((ART / "model_card.json").read_text(encoding="utf-8"))
plan_meta = json.loads((ART / "plan_meta.json").read_text(encoding="utf-8"))
evaluation = json.loads((ART / "evaluation.json").read_text(encoding="utf-8"))

RESULTS = []
def record(no, item, expected, actual, ok):
    RESULTS.append({"#": no, "점검": item, "기대": str(expected), "실제": str(actual), "결과": "O" if ok else "X"})
    print(f"[{'O' if ok else 'X'}] {no} {item} | 기대 {expected} | 실제 {actual}")

def mismatches(a, b, cols, tol=1e-9):
    # 두 표(같은 행 순서)에서 열마다 다른 칸 수. NaN끼리는 같다고 본다.
    out = {}
    for c in cols:
        x, y = a[c].reset_index(drop=True), b[c].reset_index(drop=True)
        both_na = x.isna() & y.isna()
        if pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y):
            diff = ~((x - y).abs() <= tol) & ~both_na
        else:
            diff = (x.astype(str) != y.astype(str)) & ~both_na
        out[c] = int(diff.sum())
    return out

print("ROOT =", ROOT)

ROOT = c:\workspaces\drone_data


## 1. 전처리 규칙 (AGENTS.md '데이터 처리')
- `downtime_hours` 999 → 결측(행은 지우지 않음)
- `maintenance_cost` > 10,000 → ÷100
- 예방정비의 `fault_code` · `fault_category` 결측 → `"N/A"`
- `days_since_last_maintenance` 결측 → 비워 두고 `is_first_maintenance` = 1

In [2]:
PARTS = {"train": "drone_train", "test": "drone_test", "this_month": "drone_this_month"}

def my_clean(raw):
    d = raw.copy()
    d["downtime_hours"] = d["downtime_hours"].where(d["downtime_hours"] != 999, np.nan)
    cost = d["maintenance_cost"].astype(float)
    d["maintenance_cost"] = np.where(cost > 10000, cost / 100, cost)
    pm = d["maintenance_type"] == "예방정비"
    for c in ["fault_code", "fault_category"]:
        d.loc[pm & d[c].isna(), c] = "N/A"
    d["is_first_maintenance"] = d["days_since_last_maintenance"].isna().astype(int)
    return d

raw, mine = {}, {}
for part, name in PARTS.items():
    raw[part] = read(DATA / f"{name}.csv")
    mine[part] = my_clean(raw[part])
    saved = read(ART / "clean" / f"{name}_clean.csv")
    r = raw[part]
    print(f"{part}: 999 {int((r.downtime_hours == 999).sum())}건 · 비용>10,000 {int((r.maintenance_cost > 10000).sum())}건 · "
          f"예방정비 고장 결측 {int((r.maintenance_type.eq('예방정비') & r.fault_category.isna()).sum())}건 · 첫 정비 {int(r.days_since_last_maintenance.isna().sum())}건")
    record(1, f"{part} 행 수 유지(삭제 없음)", len(r), len(saved), len(r) == len(saved))
    mm = mismatches(mine[part], saved, list(mine[part].columns))
    bad = {k: v for k, v in mm.items() if v}
    record(1, f"{part} clean 파일 = 규칙 다시 적용", "불일치 0칸", bad or "불일치 0칸", not bad)

# 전제: 고장 결측은 예방정비에만 있다 (그 외 결측이 있으면 'N/A' 채움 규칙이 맞지 않음)
odd = sum(int((r.fault_category.isna() & r.maintenance_type.ne("예방정비")).sum()) for r in raw.values())
record(1, "고장 결측 = 예방정비뿐(전제)", 0, odd, odd == 0)

train: 999 4건 · 비용>10,000 5건 · 예방정비 고장 결측 1707건 · 첫 정비 150건
[O] 1 train 행 수 유지(삭제 없음) | 기대 2346 | 실제 2346
[O] 1 train clean 파일 = 규칙 다시 적용 | 기대 불일치 0칸 | 실제 불일치 0칸
test: 999 0건 · 비용>10,000 0건 · 예방정비 고장 결측 876건 · 첫 정비 0건
[O] 1 test 행 수 유지(삭제 없음) | 기대 1241 | 실제 1241
[O] 1 test clean 파일 = 규칙 다시 적용 | 기대 불일치 0칸 | 실제 불일치 0칸
this_month: 999 0건 · 비용>10,000 0건 · 예방정비 고장 결측 62건 · 첫 정비 0건
[O] 1 this_month 행 수 유지(삭제 없음) | 기대 106 | 실제 106
[O] 1 this_month clean 파일 = 규칙 다시 적용 | 기대 불일치 0칸 | 실제 불일치 0칸
[O] 1 고장 결측 = 예방정비뿐(전제) | 기대 0 | 실제 0


## 2. 시간 피처 · 계절 (AGENTS.md: 그 정비일까지의 기록만, 같은 날 포함, train → test → this_month 이어서)
정의는 `drone_lib.add_time_features` docstring의 문장을 따르되, 코드는 쓰지 않고 **행마다 단순 필터**로 다시 센다.
- `fail_count_90d`: 90일 전 날짜 ~ 그날, 사후 · 긴급정비 수(이번 정비 포함)
- `days_since_last_failure`: 이번이 고장 정비면 0, 아니면 직전 고장 정비 뒤 경과일, 없으면 결측
- `hours_since_last_pm`: 이번이 예방정비면 0, 아니면 가장 최근 예방정비 때와의 누적 비행시간 차, 없으면 결측
- `season`: 12 ~ 2월 겨울 · 3 ~ 5 봄 · 6 ~ 8 여름 · 9 ~ 11 가을

In [3]:
FAIL = {"사후정비", "긴급정비"}
allm = pd.concat([mine[p].assign(part=p) for p in PARTS], ignore_index=True)
allm["maintenance_date"] = pd.to_datetime(allm["maintenance_date"])
allm = allm.sort_values(["equipment_id", "maintenance_date"], kind="stable").reset_index(drop=True)
allm["season"] = allm["maintenance_date"].dt.month.map(
    lambda m: "겨울" if m in (12, 1, 2) else "봄" if m <= 5 else "여름" if m <= 8 else "가을")

fc, dsf, hpm = [], [], []
for _, row in allm.iterrows():
    h = allm[(allm.equipment_id == row.equipment_id) & (allm.maintenance_date <= row.maintenance_date)]
    fails = h[h.maintenance_type.isin(FAIL)]
    fc.append(int((fails.maintenance_date >= row.maintenance_date - pd.Timedelta(days=90)).sum()))
    if row.maintenance_type in FAIL:
        dsf.append(0.0)
    else:
        dsf.append(float((row.maintenance_date - fails.maintenance_date.max()).days) if len(fails) else np.nan)
    pms = h[h.maintenance_type == "예방정비"]
    if row.maintenance_type == "예방정비":
        hpm.append(0.0)
    elif len(pms):
        last = pms.sort_values(["maintenance_date", "cumulative_operating_hours"]).iloc[-1]
        hpm.append(round(row.cumulative_operating_hours - last.cumulative_operating_hours, 1))
    else:
        hpm.append(np.nan)
allm["fail_count_90d"], allm["days_since_last_failure"], allm["hours_since_last_pm"] = fc, dsf, hpm

feat = read(ART / "features_all.csv")
feat["maintenance_date"] = pd.to_datetime(feat["maintenance_date"])
feat = feat.sort_values(["equipment_id", "maintenance_date"], kind="stable").reset_index(drop=True)
record(2, "features_all 행 수", len(allm), len(feat), len(allm) == len(feat))
keys_ok = (allm[["equipment_id", "maintenance_date", "part"]].astype(str).values ==
           feat[["equipment_id", "maintenance_date", "part"]].astype(str).values).all()
record(2, "행 순서 · 키(드론 · 정비일 · 기간) 일치", True, bool(keys_ok), bool(keys_ok))
TIME = ["season", "fail_count_90d", "days_since_last_failure", "hours_since_last_pm", "is_first_maintenance"]
mm = mismatches(allm, feat, TIME + ["downtime_hours", "maintenance_cost", "fault_category"], tol=1e-6)
bad = {k: v for k, v in mm.items() if v}
record(2, "시간 피처 · 계절 · 전처리 열 다시 계산", "불일치 0칸", bad or "불일치 0칸", not bad)
dup = int(allm.duplicated(["equipment_id", "maintenance_date"]).sum())
record(2, "같은 드론 · 같은 날 기록 0건('같은 날 포함' 전제)", 0, dup, dup == 0)

[O] 2 features_all 행 수 | 기대 3693 | 실제 3693
[O] 2 행 순서 · 키(드론 · 정비일 · 기간) 일치 | 기대 True | 실제 True
[O] 2 시간 피처 · 계절 · 전처리 열 다시 계산 | 기대 불일치 0칸 | 실제 불일치 0칸
[O] 2 같은 드론 · 같은 날 기록 0건('같은 날 포함' 전제) | 기대 0 | 실제 0


## 3. 누수 컬럼 제외 (AGENTS.md '분석 · 검증')

In [4]:
model = joblib.load(ART / "drone_model.joblib")
used = list(model.feature_names_in_)
LEAK = {"availability_rate", "equipment_id", "unit_code", "maintenance_date", "fault_code", "failure_within_30days"}
record(3, "저장된 모델 입력에 누수 · 식별 컬럼 없음", "없음", sorted(LEAK & set(used)) or "없음", not (LEAK & set(used)))
record(3, "모델 입력 = model_card features", f"{len(card['features'])}개", f"{len(used)}개 · 순서 같음 {used == card['features']}", used == card["features"])
leak_month = sorted({"availability_rate", "failure_within_30days"} & set(raw["this_month"].columns))
record(3, "이번 달 파일에 정답 · 가동률 없음", "없음", leak_month or "없음", not leak_month)
FEATURES = used

[O] 3 저장된 모델 입력에 누수 · 식별 컬럼 없음 | 기대 없음 | 실제 없음
[O] 3 모델 입력 = model_card features | 기대 17개 | 실제 17개 · 순서 같음 True
[O] 3 이번 달 파일에 정답 · 가동률 없음 | 기대 없음 | 실제 없음


## 4. 기간 분할 건수 (무작위 아님: fit ~ 2024-06-30 · 검증 2024-07 ~ 12 · 최종 학습 train 전체 · 시험 2025)

In [5]:
tr = allm[allm.part == "train"]
S = {"fit": tr[tr.maintenance_date <= "2024-06-30"],
     "val": tr[(tr.maintenance_date >= "2024-07-01") & (tr.maintenance_date <= "2024-12-31")],
     "train": tr,
     "test": allm[allm.part == "test"],
     "month": allm[allm.part == "this_month"]}
expect = {"fit": 1703, "val": 643, "train": 2346, "test": 1241, "month": 106}  # model_card split 문자열 · plan_meta records
for k, v in expect.items():
    record(4, f"{k} 건수", v, len(S[k]), len(S[k]) == v)
gap = (S["fit"].maintenance_date.max() < S["val"].maintenance_date.min()) and (tr.maintenance_date.max() < S["test"].maintenance_date.min())
record(4, "기간이 겹치지 않음(fit < val, train < test)", True, bool(gap), bool(gap))

[O] 4 fit 건수 | 기대 1703 | 실제 1703
[O] 4 val 건수 | 기대 643 | 실제 643
[O] 4 train 건수 | 기대 2346 | 실제 2346
[O] 4 test 건수 | 기대 1241 | 실제 1241
[O] 4 month 건수 | 기대 106 | 실제 106
[O] 4 기간이 겹치지 않음(fit < val, train < test) | 기대 True | 실제 True


## 5. 검증 성능 · 월 20대 적중
- 저장된 모델과 **같은 설정을 복제(clone)**해 fit 기간으로 다시 학습 → 검증 확률을 `val_predictions.csv`와 대조
- PR-AUC · ROC-AUC를 sklearn으로 다시 계산 → model_card `validation`과 대조
- 월 20대 적중(드론 단위): 그 달 드론별 최근 기록 하나 → 확률 상위 20대 중 실제 재고장 수

In [6]:
TARGET = "failure_within_30days"
m_val = clone(model).fit(S["fit"][FEATURES], S["fit"][TARGET].astype(int))
val = S["val"].copy()
val["prob_mine"] = m_val.predict_proba(val[FEATURES])[:, 1]
vp = read(ART / "val_predictions.csv")
record(5, "검증 행 수", len(vp), len(val), len(vp) == len(val))
d = float(np.abs(val["prob_mine"].to_numpy() - vp["prob"].to_numpy()).max())
record(5, "다시 학습한 검증 확률 = val_predictions.csv", "최대 차이 < 1e-6", f"{d:.1e}", d < 1e-6)
lab = int((val[TARGET].astype(int).to_numpy() != vp[TARGET].astype(int).to_numpy()).sum())
record(5, "검증 정답 = 원본 train 정답", 0, lab, lab == 0)

y, p = vp[TARGET].astype(int), vp["prob"]
V = card["validation"]
for name, mine_v, exp in [("PR-AUC", round(average_precision_score(y, p), 3), V["pr_auc"]),
                          ("ROC-AUC", round(roc_auc_score(y, p), 3), V["roc_auc"]),
                          ("재고장 비율", round(y.mean(), 3), V["base_rate"]),
                          ("양성 수", int(y.sum()), V["positives"])]:
    record(5, f"검증 {name}", exp, mine_v, mine_v == exp)

def month_top20(df, score, ycol, k=20):
    # 동점 규칙: 드론 번호 순으로 놓고 앞쪽 우선(drone-plan과 같은 기준). 나이(오래된 순)는 동점이 많아 이 규칙이 결과를 정한다.
    hits = fails = 0
    for _, g in df.groupby(df.maintenance_date.dt.to_period("M")):
        g = g.sort_values("maintenance_date", kind="stable").drop_duplicates("equipment_id", keep="last")
        g = g.sort_values("equipment_id", kind="stable")
        hits += int(g.nlargest(k, score)[ycol].sum()); fails += int(g[ycol].sum())
    return hits, fails

vv = vp.assign(maintenance_date=pd.to_datetime(vp.maintenance_date), y=vp[TARGET].astype(int))
h_model, f_all = month_top20(vv, "prob", "y")
h_old, _ = month_top20(vv, "equipment_age_years", "y")
T = V["top20_hits_6m"]
record(5, "검증 6개월 20대 적중 (모델 · 오래된 순 · 전체 재고장)", [T["model"], T["oldest"], T["failures"]],
       [h_model, h_old, f_all], [h_model, h_old, f_all] == [T["model"], T["oldest"], T["failures"]])

[O] 5 검증 행 수 | 기대 643 | 실제 643


[O] 5 다시 학습한 검증 확률 = val_predictions.csv | 기대 최대 차이 < 1e-6 | 실제 0.0e+00
[O] 5 검증 정답 = 원본 train 정답 | 기대 0 | 실제 0


[O] 5 검증 PR-AUC | 기대 0.776 | 실제 0.776
[O] 5 검증 ROC-AUC | 기대 0.879 | 실제 0.879
[O] 5 검증 재고장 비율 | 기대 0.274 | 실제 0.274
[O] 5 검증 양성 수 | 기대 176 | 실제 176
[O] 5 검증 6개월 20대 적중 (모델 · 오래된 순 · 전체 재고장) | 기대 [38.0, 18.0, 56.0] | 실제 [38, 18, 56]


## 6. 임계값 선택 (검증 기간 비용 최소: 놓친 재고장 X만원 × 건수 + 재점검 60만원 × 기준 이상 건수, 후보 0.05 ~ 0.94)

In [7]:
def best_threshold(y, p, miss, check=60):
    best = None
    for t in np.round(np.arange(0.05, 0.95, 0.01), 2):
        flag = p >= t
        missed = int((~flag & (y == 1)).sum()); flagged = int(flag.sum())
        cost = missed * miss + flagged * check
        if best is None or cost < best["cost"]:   # 같은 비용이면 낮은 임계값(먼저 나온 값)
            best = {"threshold": float(t), "flagged": flagged, "missed": missed, "cost": cost}
    return best

for miss, exp in card["threshold_by_miss_cost"].items():
    b = best_threshold(y.to_numpy(), p.to_numpy(), int(miss))
    record(6, f"놓친 재고장 {miss}만원 기준", exp, b, b == exp)
record(6, "채택 임계값(300만원) = model_card threshold", card["threshold"],
       best_threshold(y.to_numpy(), p.to_numpy(), 300)["threshold"],
       best_threshold(y.to_numpy(), p.to_numpy(), 300)["threshold"] == card["threshold"])
THR = card["threshold"]

[O] 6 놓친 재고장 150만원 기준 | 기대 {'threshold': 0.35, 'flagged': 187, 'missed': 46, 'cost': 18120} | 실제 {'threshold': 0.35, 'flagged': 187, 'missed': 46, 'cost': 18120}
[O] 6 놓친 재고장 300만원 기준 | 기대 {'threshold': 0.22, 'flagged': 245, 'missed': 28, 'cost': 23100} | 실제 {'threshold': 0.22, 'flagged': 245, 'missed': 28, 'cost': 23100}
[O] 6 놓친 재고장 600만원 기준 | 기대 {'threshold': 0.12, 'flagged': 309, 'missed': 20, 'cost': 30540} | 실제 {'threshold': 0.12, 'flagged': 309, 'missed': 20, 'cost': 30540}
[O] 6 채택 임계값(300만원) = model_card threshold | 기대 0.22 | 실제 0.22


## 7. 시험 예측 (제출물 1 · `outputs/test_predictions_2025.csv`)
- 형식: 열 이름 · 행 수 · 드론 · 정비일 조합 · 날짜 형식 · 확률 범위
- 저장된 모델로 다시 예측 → 제출 확률과 대조
- 같은 설정으로 train 전체에서 **다시 학습** → 저장된 모델이 정말 train 전체(2023 ~ 2024)로 학습됐는지 확인

In [8]:
sub = pd.read_csv(OUT / "test_predictions_2025.csv", dtype={"maintenance_date": str})
rt = raw["test"]
record(7, "열 이름", ["equipment_id", "maintenance_date", "prob"], list(sub.columns), list(sub.columns) == ["equipment_id", "maintenance_date", "prob"])
record(7, "행 수 = drone_test.csv", len(rt), len(sub), len(rt) == len(sub))
same_keys = set(zip(sub.equipment_id, sub.maintenance_date)) == set(zip(rt.equipment_id, rt.maintenance_date.astype(str)))
record(7, "드론 · 정비일 조합 = drone_test.csv", True, same_keys, same_keys)
fmt = bool(sub.maintenance_date.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all())
rng = bool(sub.prob.notna().all() and sub.prob.between(0, 1).all())
record(7, "날짜 YYYY-MM-DD · 확률 결측 없음 · 0 ~ 1", True, fmt and rng, fmt and rng)

test = S["test"].copy()
test["key"] = test.equipment_id + "|" + test.maintenance_date.dt.strftime("%Y-%m-%d")
sub["key"] = sub.equipment_id + "|" + sub.maintenance_date
test = test.merge(sub[["key", "prob"]], on="key")
p_saved = model.predict_proba(test[FEATURES])[:, 1]
d1 = float(np.abs(p_saved.round(6) - test.prob).max())
record(7, "저장 모델 다시 예측 = 제출 확률", "최대 차이 < 1e-6", f"{d1:.1e}", d1 < 1e-6)
m_full = clone(model).fit(S["train"][FEATURES], S["train"][TARGET].astype(int))
d2 = float(np.abs(m_full.predict_proba(test[FEATURES])[:, 1] - p_saved).max())
record(7, "train 전체로 다시 학습 = 저장 모델", "최대 차이 < 1e-6", f"{d2:.1e}", d2 < 1e-6)
mean_ok = round(float(sub.prob.mean()), 3) == card["submission"]["prob_mean"]
record(7, "제출 평균 확률 · 기준 이상 건수", [card["submission"]["prob_mean"], card["submission"]["flag_ge_threshold"]],
       [round(float(sub.prob.mean()), 3), int((sub.prob >= THR).sum())],
       mean_ok and int((sub.prob >= THR).sum()) == card["submission"]["flag_ge_threshold"])

[O] 7 열 이름 | 기대 ['equipment_id', 'maintenance_date', 'prob'] | 실제 ['equipment_id', 'maintenance_date', 'prob']
[O] 7 행 수 = drone_test.csv | 기대 1241 | 실제 1241
[O] 7 드론 · 정비일 조합 = drone_test.csv | 기대 True | 실제 True
[O] 7 날짜 YYYY-MM-DD · 확률 결측 없음 · 0 ~ 1 | 기대 True | 실제 True
[O] 7 저장 모델 다시 예측 = 제출 확률 | 기대 최대 차이 < 1e-6 | 실제 0.0e+00
[O] 7 train 전체로 다시 학습 = 저장 모델 | 기대 최대 차이 < 1e-6 | 실제 0.0e+00
[O] 7 제출 평균 확률 · 기준 이상 건수 | 기대 [0.262, 451] | 실제 [0.262, 451]


## 8. 2025 평가 (정답 재현)
README 정의 "정비 뒤 1 ~ 30일 안에 같은 드론의 사후 · 긴급정비가 있으면 1"을 직접 구현한다.
- 먼저 **train(정답 있음)**에서 이 규칙이 실제 정답과 100% 맞는지 확인 → 맞아야 2025에 써도 된다
- 2025 PR-AUC · 20대 적중을 다시 계산해 `evaluation.json`과 대조
- 이 정답은 강사 채점과 다를 수 있다(재현값). 모델 · 임계값 선택에는 쓰지 않았다.

In [9]:
def rebuild(df):
    y = np.zeros(len(df), dtype=int)
    for i, (_, r) in enumerate(df.iterrows()):
        later = df[(df.equipment_id == r.equipment_id) & df.maintenance_type.isin(FAIL)]
        gap = (later.maintenance_date - r.maintenance_date).dt.days
        y[i] = int(((gap >= 1) & (gap <= 30)).any())
    return y

allm["y_rebuilt"] = rebuild(allm)
trm = allm[allm.part == "train"]
rate = float((trm.y_rebuilt == trm[TARGET].astype(int)).mean())
record(8, "재현 규칙 vs train 실제 정답 일치율", evaluation["rebuild_rule_match_train"], round(rate, 4), rate == evaluation["rebuild_rule_match_train"])

te = S["test"].copy()
te["key"] = te.equipment_id + "|" + te.maintenance_date.dt.strftime("%Y-%m-%d")
te = te.merge(sub[["key", "prob"]], on="key").merge(allm[["equipment_id", "maintenance_date", "y_rebuilt"]], on=["equipment_id", "maintenance_date"])
E = evaluation["test_2025"]
for name, mine_v, exp in [("PR-AUC", round(average_precision_score(te.y_rebuilt, te.prob), 3), E["pr_auc"]),
                          ("ROC-AUC", round(roc_auc_score(te.y_rebuilt, te.prob), 3), E["roc_auc"]),
                          ("양성 수", int(te.y_rebuilt.sum()), E["positives"])]:
    record(8, f"2025 {name}", exp, mine_v, mine_v == exp)
h_m, f_t = month_top20(te, "prob", "y_rebuilt")
h_o, _ = month_top20(te, "equipment_age_years", "y_rebuilt")
T = E["top20_hits_12m"]
record(8, "2025 12개월 20대 적중 (모델 · 오래된 순 · 전체 재고장)", [T["model"], T["oldest"], T["failures"]],
       [h_m, h_o, f_t], [h_m, h_o, f_t] == [T["model"], T["oldest"], T["failures"]])

[O] 8 재현 규칙 vs train 실제 정답 일치율 | 기대 1.0 | 실제 1.0
[O] 8 2025 PR-AUC | 기대 0.741 | 실제 0.741
[O] 8 2025 ROC-AUC | 기대 0.849 | 실제 0.849
[O] 8 2025 양성 수 | 기대 293 | 실제 293
[O] 8 2025 12개월 20대 적중 (모델 · 오래된 순 · 전체 재고장) | 기대 [63.0, 28.0, 96.0] | 실제 [63, 28, 96]


## 9. 재점검 20대 (`artifacts/score_2026-01.csv`)
이번 달 드론별 최근 기록 하나 → 저장 모델 확률 → 높은 순 순위. 임계값 이상 대수 · 90일 고장 수도 대조한다.

In [10]:
mon = S["month"].sort_values("maintenance_date", kind="stable").drop_duplicates("equipment_id", keep="last")
mon = mon.sort_values("equipment_id", kind="stable").copy()
mon["prob"] = model.predict_proba(mon[FEATURES])[:, 1]
mon = mon.sort_values("prob", ascending=False, kind="stable").reset_index(drop=True)
mon["rank"] = mon.index + 1
score = read(ART / "score_2026-01.csv")
record(9, "평가 드론 수", plan_meta["drones"], len(mon), len(mon) == plan_meta["drones"] == len(score))
same_order = mon.equipment_id.tolist() == score.equipment_id.tolist()
record(9, "81대 순위(드론 순서) 같음", True, same_order, same_order)
top_same = mon.equipment_id[:20].tolist() == score.equipment_id[:20].tolist()
record(9, "상위 20대 같음", True, top_same, top_same)
dp = float(np.abs(mon.prob.round(4) - score.prob).max())
record(9, "확률(소수 4자리) = score 파일", "최대 차이 0", dp, dp == 0)
over = int((mon.prob >= THR).sum())
record(9, f"임계값 {THR} 이상 대수", plan_meta["over_threshold"], over, over == plan_meta["over_threshold"] == int(score.over_threshold.sum()))
fcm = int((mon.fail_count_90d.to_numpy() != score.fail_count_90d.to_numpy()).sum())
record(9, "90일 고장 수 = score 파일", "불일치 0", fcm, fcm == 0)
print(mon[["rank", "equipment_id", "unit_code", "fail_count_90d", "prob"]].head(5).to_string(index=False))

[O] 9 평가 드론 수 | 기대 81 | 실제 81
[O] 9 81대 순위(드론 순서) 같음 | 기대 True | 실제 True
[O] 9 상위 20대 같음 | 기대 True | 실제 True
[O] 9 확률(소수 4자리) = score 파일 | 기대 최대 차이 0 | 실제 0.0
[O] 9 임계값 0.22 이상 대수 | 기대 33 | 실제 33
[O] 9 90일 고장 수 = score 파일 | 기대 불일치 0 | 실제 0
 rank equipment_id unit_code  fail_count_90d     prob
    1      DR-0115   UNIT-02              11 0.943442
    2      DR-0100   UNIT-02               3 0.815321
    3      DR-0038   UNIT-10               5 0.811233
    4      DR-0046   UNIT-12               4 0.777758
    5      DR-0017   UNIT-06               4 0.695580


## 10. 앱 데이터 — Supabase `public.drone_score_2026_01` = CSV (읽기만)

In [11]:
try:
    import sqlalchemy as sa
    c = tomllib.loads((ROOT / ".streamlit" / "secrets.toml").read_text(encoding="utf-8"))["connections"]["supabase"]
    url = sa.URL.create(f"{c['dialect']}+{c['driver']}" if c.get("driver") else c["dialect"], username=c["username"],
                        password=c["password"], host=c["host"], port=int(c["port"]), database=c["database"], query=c.get("query", {}))
    with sa.create_engine(url, connect_args={"connect_timeout": 15}).connect() as con:
        db = pd.read_sql('select * from public."drone_score_2026_01" order by rank', con)
    db["maintenance_date"] = pd.to_datetime(db.maintenance_date).dt.strftime("%Y-%m-%d")
    record(10, "DB 행 수", len(score), len(db), len(db) == len(score))
    mm = mismatches(db, score, list(score.columns), tol=0)
    bad = {k: v for k, v in mm.items() if v}
    record(10, "DB 값 = score_2026-01.csv (12개 열)", "불일치 0칸", bad or "불일치 0칸", not bad)
    fp = hashlib.sha256((ART / "score_2026-01.csv").read_bytes()).hexdigest()[:12]
    record(10, "DB 지문 열 = 현재 score 파일 지문", fp, db.score_fp.unique().tolist(), db.score_fp.unique().tolist() == [fp])
except Exception as e:
    record(10, "DB 대조", "연결", f"대조 불가: {type(e).__name__}", False)

[O] 10 DB 행 수 | 기대 81 | 실제 81
[O] 10 DB 값 = score_2026-01.csv (12개 열) | 기대 불일치 0칸 | 실제 불일치 0칸
[O] 10 DB 지문 열 = 현재 score 파일 지문 | 기대 1aedddc723ca | 실제 ['1aedddc723ca']


## 11. 지문 연결 — 결과 파일이 어떤 데이터 · 모델에서 나왔는지 (SHA-256 앞 12자리)

In [12]:
fp = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()[:12]
now = {"train": fp(DATA / "drone_train.csv"), "test": fp(DATA / "drone_test.csv"), "this_month": fp(DATA / "drone_this_month.csv")}
record(11, "현재 데이터 = 학습 때(model_card)", card["lineage"]["data"], now, now == card["lineage"]["data"])
record(11, "현재 데이터 = 계획 때(plan_meta)", plan_meta["data"], now, now == plan_meta["data"])
record(11, "모델 파일 = model_card · plan_meta", card["lineage"]["model"], fp(ART / "drone_model.joblib"),
       fp(ART / "drone_model.joblib") == card["lineage"]["model"] == plan_meta["model"])
record(11, "제출 파일 = model_card · evaluation", card["lineage"]["submission"], fp(OUT / "test_predictions_2025.csv"),
       fp(OUT / "test_predictions_2025.csv") == card["lineage"]["submission"] == evaluation["lineage"]["submission"])
record(11, "score 파일 = plan_meta", plan_meta["score"], fp(ART / "score_2026-01.csv"), fp(ART / "score_2026-01.csv") == plan_meta["score"])

[O] 11 현재 데이터 = 학습 때(model_card) | 기대 {'train': '26c4d3711fd5', 'test': '5fa789dd8111', 'this_month': '379631f03af5'} | 실제 {'train': '26c4d3711fd5', 'test': '5fa789dd8111', 'this_month': '379631f03af5'}
[O] 11 현재 데이터 = 계획 때(plan_meta) | 기대 {'train': '26c4d3711fd5', 'test': '5fa789dd8111', 'this_month': '379631f03af5'} | 실제 {'train': '26c4d3711fd5', 'test': '5fa789dd8111', 'this_month': '379631f03af5'}
[O] 11 모델 파일 = model_card · plan_meta | 기대 fa51be49c230 | 실제 fa51be49c230
[O] 11 제출 파일 = model_card · evaluation | 기대 5d5a4ee5dd39 | 실제 5d5a4ee5dd39
[O] 11 score 파일 = plan_meta | 기대 1aedddc723ca | 실제 1aedddc723ca


## 12. 결함 주입 — 위 점검이 '항상 O'가 아니라 실제로 틀린 값을 잡는지
메모리 복사본만 바꾼다(파일은 그대로). **점검이 X를 내야 통과(O)**다.

In [13]:
# (a) 전처리: 비용 자릿수 실수 1건을 ÷100 하지 않은 상태
c_bad = read(ART / "clean" / "drone_train_clean.csv")
i = raw["train"].index[raw["train"].maintenance_cost > 10000][0]
c_bad.loc[i, "maintenance_cost"] = raw["train"].loc[i, "maintenance_cost"]
n = mismatches(mine["train"], c_bad, ["maintenance_cost"])["maintenance_cost"]
record(12, "(a) 비용 ÷100 누락 1건 → 1번 점검이 잡음", "불일치 1", n, n == 1)

# (b) 시간 피처: 90일 고장 수 1칸 +1
f_bad = feat.copy(); f_bad.loc[100, "fail_count_90d"] += 1
n = mismatches(allm, f_bad, ["fail_count_90d"])["fail_count_90d"]
record(12, "(b) 90일 고장 수 1칸 변경 → 2번 점검이 잡음", "불일치 1", n, n == 1)

# (c) 재점검: 21위 드론 확률을 1위보다 높게 → 순위 · 확률 대조가 잡음
s_bad = score.copy(); s_bad.loc[20, "prob"] = 0.99
s_bad = s_bad.sort_values("prob", ascending=False, kind="stable").reset_index(drop=True)
caught = (mon.equipment_id.tolist() != s_bad.equipment_id.tolist()) and float(np.abs(mon.prob.round(4).values - s_bad.prob.values).max()) > 0
record(12, "(c) score 확률 1개 조작 → 9번 점검이 잡음", True, caught, caught)

# (d) 임계값: 비용 규칙의 재점검 비용을 60 → 30으로 잘못 쓰면 다른 임계값이 나옴
t_bad = best_threshold(y.to_numpy(), p.to_numpy(), 300, check=30)["threshold"]
record(12, "(d) 비용 규칙 오류 → 6번 점검이 잡음", f"≠ {THR}", t_bad, t_bad != THR)

# (e) 누수: 가동률을 넣은 모델이었다면 3번 점검이 잡음
leak_used = FEATURES + ["availability_rate"]
record(12, "(e) 누수 컬럼 추가 → 3번 점검이 잡음", "availability_rate 검출", sorted(LEAK & set(leak_used)), bool(LEAK & set(leak_used)))

[O] 12 (a) 비용 ÷100 누락 1건 → 1번 점검이 잡음 | 기대 불일치 1 | 실제 1
[O] 12 (b) 90일 고장 수 1칸 변경 → 2번 점검이 잡음 | 기대 불일치 1 | 실제 1
[O] 12 (c) score 확률 1개 조작 → 9번 점검이 잡음 | 기대 True | 실제 True
[O] 12 (d) 비용 규칙 오류 → 6번 점검이 잡음 | 기대 ≠ 0.22 | 실제 0.12
[O] 12 (e) 누수 컬럼 추가 → 3번 점검이 잡음 | 기대 availability_rate 검출 | 실제 ['availability_rate']


## 종합

In [14]:
res = pd.DataFrame(RESULTS)
print(f"통과 {int((res['결과'] == 'O').sum())} / {len(res)}  ·  X {int((res['결과'] == 'X').sum())}건")
res.groupby("#")["결과"].apply(lambda s: f"{(s == 'O').sum()}/{len(s)}").to_frame("통과").T

통과 63 / 63  ·  X 0건


#,1,2,3,4,5,6,7,8,9,10,11,12
통과,7/7,4/4,3/3,6/6,8/8,4/4,7/7,5/5,6/6,3/3,5/5,5/5


In [15]:
res[res["결과"] == "X"] if (res["결과"] == "X").any() else print("X 항목 없음")

X 항목 없음


### 이 노트북이 보장하지 않는 것
- **2026-01 20대가 실제로 재고장했는지**: 2026-02 기록이 없어 알 수 없다.
- **2025 정답**: README 정의로 재현한 값이다(train 일치율로 규칙을 확인). 강사 채점과 다를 수 있다.
- **모델 설정 자체의 타당성**(하이퍼파라미터 · 피처 선택이 좋은가): 이 노트북은 '정해진 규칙대로 계산됐는가'만 본다.
- 같은 사람이 규칙을 읽고 다시 구현했으므로, **규칙 문장 자체를 잘못 이해했다면** 두 구현이 같이 틀릴 수 있다. 이 위험은 2번 손 확인(예: DR-0115 90일 고장 11회)과 8번 train 정답 일치율로 줄인다.